# Estimating a population mean with the CLT: Titanic fares
All 1309 Titanic passengers (train + test files) are the population. We pretend not to know the average fare, estimate it from samples with the central limit theorem, and compare with the truth. Data: Kaggle Titanic `train.csv` and `test.csv`, from the `titanic-kaggle-submission` repository of campusx-official on GitHub.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## 1. The population

In [ ]:
train = pd.read_csv("data/titanic_train.csv")
test = pd.read_csv("data/titanic_test.csv")
print(train.shape, test.shape)             # test has no Survived column
# stack both files into one table of every passenger, then shuffle the rows
df = pd.concat([train.drop(columns="Survived"), test]).sample(frac=1, random_state=42)
print(len(df), "passengers")
fare = df["Fare"].dropna()                  # one fare is missing
print(len(fare), "known fares")
print("skewness", round(fare.skew(), 2), " median", fare.median(), " max", fare.max())

## 2. 100 samples of 50 passengers

In [ ]:
rng = np.random.default_rng(42)              # fixed seed
# each sample: 50 different passengers (drawn without replacement)
samples = np.array([fare.sample(50, random_state=rng).to_numpy() for _ in range(100)])
print(samples.shape)                        # 100 samples, 50 fares each
sample_means = samples.mean(axis=1)         # the sampling distribution of the mean
print("first five sample means", sample_means[:5].round(2))
print("skewness of the sample means", round(stats.skew(sample_means), 2))

## 3. Point estimate and standard error

In [ ]:
grand_mean = sample_means.mean()            # average of the 100 sample means
sd_means = sample_means.std(ddof=1)         # spread of the sample means
print("average of the sample means:", round(grand_mean, 2))
print("std of the sample means:", round(sd_means, 2))
# the CLT says this std is sigma / sqrt(50); check with the (normally unknown) population sigma
sigma = fare.std(ddof=0)
print("sigma / sqrt(50) =", round(sigma / np.sqrt(50), 2))

## 4. A range for the population mean: average ± 2 standard errors
The average of 100 sample means has standard error `sd_means / sqrt(100)`, because it averages 100 independent sample means.

In [ ]:
se_grand = sd_means / np.sqrt(100)
print("standard error of the average:", round(se_grand, 3))
for k, share in [(1, "68%"), (2, "95%"), (3, "99.7%")]:
    print(f"±{k} SE ({share}): {grand_mean - k * se_grand:.2f} to {grand_mean + k * se_grand:.2f}")
# the wrong divisor: sqrt(50), the sample size, instead of sqrt(100), the number of samples
wrong = sd_means / np.sqrt(50)
print("with sqrt(50):", round(grand_mean - 2 * wrong, 2), "to", round(grand_mean + 2 * wrong, 2))

## 5. The truth

In [ ]:
print("population mean fare:", round(fare.mean(), 2))

## 6. With a single sample of 50
In practice we have one sample: x̄ ± 2 s / √n.

In [ ]:
x = samples[0]
s = x.std(ddof=1)
print("one sample: mean", round(x.mean(), 2), " s", round(s, 2))
print("range:", round(x.mean() - 2 * s / np.sqrt(50), 2), "to", round(x.mean() + 2 * s / np.sqrt(50), 2))

## 7. How often does each method catch the true mean? 1000 repetitions

In [ ]:
F, mu = fare.to_numpy(), fare.mean()
hits = {"sqrt(50) divisor": 0, "sqrt(100) divisor": 0, "one sample, n = 50": 0}
for r in range(1000):
    rr = np.random.default_rng(1000 + r)
    S = np.array([rr.choice(F, 50, replace=False) for _ in range(100)])
    m = S.mean(axis=1)
    g, sd = m.mean(), m.std(ddof=1)
    hits["sqrt(50) divisor"] += abs(g - mu) <= 2 * sd / np.sqrt(50)
    hits["sqrt(100) divisor"] += abs(g - mu) <= 2 * sd / np.sqrt(100)
    one = S[0]
    hits["one sample, n = 50"] += abs(one.mean() - mu) <= 2 * one.std(ddof=1) / np.sqrt(50)
for name, h in hits.items():
    print(f"{name:20s} caught the true mean {h / 10:.1f}% of the time")

## 8. Bigger single samples help with very skewed data

In [ ]:
for n in [50, 200]:
    hit = 0
    for r in range(2000):
        one = np.random.default_rng(5000 + r).choice(F, n, replace=False)
        hit += abs(one.mean() - mu) <= 2 * one.std(ddof=1) / np.sqrt(n)
    print(f"one sample, n = {n}: {hit / 20:.1f}%")

## 9. Template: the average income of a country (simulated data)
We do not have real salary data, so we simulate a log-normal population of one million yearly incomes in rupees.

In [ ]:
rng = np.random.default_rng(0)
incomes = rng.lognormal(mean=np.log(100_000), sigma=0.7, size=1_000_000)   # the population
# step 1: several random samples, each with n >= 30
k, n = 100, 50
income_samples = rng.choice(incomes, size=(k, n))
# step 2: the mean of each sample
means = income_samples.mean(axis=1)
# step 3: the average of the sample means = point estimate
estimate = means.mean()
# step 4: standard error of that average = std of the sample means / sqrt(number of samples)
se = means.std(ddof=1) / np.sqrt(k)
# step 5: 95% range with z = 1.96
low, high = estimate - 1.96 * se, estimate + 1.96 * se
print(f"estimate {estimate:,.0f} rupees, 95% range {low:,.0f} to {high:,.0f}")
print(f"true population mean {incomes.mean():,.0f} rupees")